<a href="https://colab.research.google.com/github/Rakshitanaik-27/EliteTech-DataScience/blob/main/Task3_End_To_End_WebApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Train Diabetes Model and Save File
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 1. Load Pima Indians Diabetes Dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
col_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']
df = pd.read_csv(url, names=col_names)

X = df.drop('Outcome', axis=1)
y = df['Outcome']

# 2. Train Random Forest Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 3. Save Model
joblib.dump(model, "diabetes_model.pkl")
print("✅ 'diabetes_model.pkl' created successfully in your Colab files!")

✅ 'diabetes_model.pkl' created successfully in your Colab files!


In [7]:
%%writefile app.py
import streamlit as st
import joblib
import numpy as np
import pandas as pd
import datetime

# ------------------------------------------------------------------------------
# PAGE CONFIGURATION
# ------------------------------------------------------------------------------
st.set_page_config(
    page_title="Diabetes Risk Assessment",
    page_icon="🩺",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for UI styling
st.markdown("""
    <style>
    .main { background-color: #f8fafc; }
    .metric-card {
        background-color: #ffffff;
        padding: 18px;
        border-radius: 12px;
        box-shadow: 0 1px 3px rgba(0,0,0,0.05);
        border: 1px solid #e2e8f0;
        text-align: center;
    }
    .metric-title { color: #64748b; font-size: 14px; font-weight: 600; margin-bottom: 6px; }
    .metric-value { font-size: 24px; font-weight: 700; color: #0f172a; }
    .profile-box {
        background-color: #ffffff;
        padding: 20px;
        border-radius: 12px;
        border: 1px solid #e2e8f0;
        box-shadow: 0 1px 3px rgba(0,0,0,0.05);
    }
    </style>
""", unsafe_allow_html=True)

# ------------------------------------------------------------------------------
# MODEL LOADING & SESSION STATE INITIALIZATION
# ------------------------------------------------------------------------------
@st.cache_resource
def load_model():
    return joblib.load('diabetes_model.pkl')

model = load_model()

# Initialize session state for prediction history
if 'history' not in st.session_state:
    st.session_state.history = pd.DataFrame(columns=["Date", "Patient Name", "Risk %", "Result", "Confidence"])

# ------------------------------------------------------------------------------
# SIDEBAR CONTROLS
# ------------------------------------------------------------------------------
st.sidebar.title("🩺 Control Panel")
st.sidebar.markdown("---")
st.sidebar.subheader("Patient Health Profile")

patient_name = st.sidebar.text_input("Patient Name", "Rakshita Naik")
gender = st.sidebar.selectbox("Gender", ["Female", "Male"])
age = st.sidebar.slider("Age (Years)", 21, 81, 69)
pregnancies = st.sidebar.slider("Pregnancies", 0, 17, 1)
glucose = st.sidebar.slider("Glucose Level (mg/dL)", 0, 200, 140)
blood_pressure = st.sidebar.slider("Blood Pressure (mm Hg)", 0, 122, 115)
skin_thickness = st.sidebar.slider("Skin Thickness (mm)", 0, 99, 22)
insulin = st.sidebar.slider("Insulin Level (mu U/ml)", 0, 846, 263)
bmi = st.sidebar.slider("BMI", 0.0, 67.1, 56.03)
pedigree = st.sidebar.slider("Diabetes Pedigree Function", 0.078, 2.42, 0.372)

st.sidebar.markdown("---")
analyze_btn = st.sidebar.button("🔍 Analyze & Save Record", type="primary", use_container_width=True)

# ------------------------------------------------------------------------------
# MACHINE LEARNING INFERENCE
# ------------------------------------------------------------------------------
input_data = np.array([[pregnancies, glucose, blood_pressure, skin_thickness, insulin, bmi, pedigree, age]])
prediction = model.predict(input_data)[0]
probabilities = model.predict_proba(input_data)[0]
risk_score = probabilities[1] * 100
confidence = max(probabilities) * 100
result_label = "High Risk" if prediction == 1 else "Low Risk"

# Save record when user clicks the analyze button
if analyze_btn:
    today_str = datetime.date.today().strftime('%d %b %Y')
    new_record = pd.DataFrame([{
        "Date": today_str,
        "Patient Name": patient_name,
        "Risk %": f"{risk_score:.1f}%",
        "Result": result_label,
        "Confidence": f"{confidence:.1f}%"
    }])
    st.session_state.history = pd.concat([new_record, st.session_state.history], ignore_index=True)

# ------------------------------------------------------------------------------
# MAIN DASHBOARD LAYOUT
# ------------------------------------------------------------------------------
st.title("Diabetes Risk Assessment")
st.caption("Machine Learning based prediction using health parameters")

# ROW 1: TOP KPI METRIC CARDS
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.markdown(f"""
        <div class="metric-card">
            <div class="metric-title">🩸 Glucose</div>
            <div class="metric-value">{glucose} <span style="font-size:14px; color:#64748b;">mg/dL</span></div>
        </div>
    """, unsafe_allow_html=True)

with col2:
    st.markdown(f"""
        <div class="metric-card">
            <div class="metric-title">🫀 Blood Pressure</div>
            <div class="metric-value">{blood_pressure} <span style="font-size:14px; color:#64748b;">mm Hg</span></div>
        </div>
    """, unsafe_allow_html=True)

with col3:
    st.markdown(f"""
        <div class="metric-card">
            <div class="metric-title">⚖️ BMI</div>
            <div class="metric-value">{bmi}</div>
        </div>
    """, unsafe_allow_html=True)

with col4:
    st.markdown(f"""
        <div class="metric-card">
            <div class="metric-title">💉 Insulin</div>
            <div class="metric-value">{insulin} <span style="font-size:14px; color:#64748b;">µU/mL</span></div>
        </div>
    """, unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

# ROW 2: PREDICTION RESULT & PATIENT PROFILE
res_col1, res_col2 = st.columns([1.5, 1])

with res_col1:
    st.subheader("Prediction Result")
    p_col1, p_col2 = st.columns([1, 2])

    with p_col1:
        st.metric(label="Risk Score", value=f"{risk_score:.1f}%")
        if prediction == 1:
            st.error("High Risk")
        else:
            st.success("Low Risk")

    with p_col2:
        st.write(f"**Status:** {'High chance of developing diabetes' if prediction == 1 else 'Low chance of developing diabetes'}")
        st.write("Based on the provided health parameters, the machine learning model has evaluated patient health risk factors.")
        st.write(f"**Model Confidence:** `{confidence:.1f}%`")

with res_col2:
    st.subheader("Patient Profile")
    st.markdown(f"""
        <div class="profile-box">
            <p><strong>Name:</strong> {patient_name}</p>
            <p><strong>Age:</strong> {age} years</p>
            <p><strong>Gender:</strong> {gender}</p>
            <p><strong>BMI:</strong> {bmi}</p>
            <p><strong>Last Updated:</strong> {datetime.date.today().strftime('%d %b %Y')}</p>
        </div>
    """, unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

# ROW 3: FEATURE IMPORTANCE & PREDICTION HISTORY
hist_col1, hist_col2 = st.columns([1.2, 1.8])

with hist_col1:
    st.subheader("Feature Importance")

    # Extract real feature importances from trained model
    importances = model.feature_importances_
    features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Pedigree', 'Age']
    df_feat = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=True)

    st.bar_chart(df_feat.set_index('Feature'))

with hist_col2:
    st.subheader("Prediction History")
    if not st.session_state.history.empty:
        st.dataframe(st.session_state.history, use_container_width=True)
    else:
        st.info("No records saved yet. Adjust sliders and click 'Analyze & Save Record' in the sidebar.")

Overwriting app.py


In [9]:
# Cell 3: Clean Streamlit Launcher (Fixes JavaScript Asset Errors)

!pip install streamlit -q

# 1. Download and set up Cloudflare Tunnel executable
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 2. Terminate any previous hanging processes
!pkill streamlit
!pkill cloudflared

import subprocess
import re
import time

# 3. Launch Streamlit server with asset protection disabled
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false",
    "--server.headless=true",
    "--server.port=8501"
])

print("⏳ Launching Streamlit background engine...")
time.sleep(5)

# 4. Create public secure tunnel
process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("⏳ Fetching active live domain...")
time.sleep(3)

for line in process.stderr:
    if "trycloudflare.com" in line:
        url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if url_match:
            live_url = url_match.group(0)
            print("\n" + "="*60)
            print(f"🚀 YOUR WORKING DASHBOARD URL: {live_url}")
            print("="*60 + "\n")
            print("Click the link above to test your dynamic dashboard!")
            break

⏳ Launching Streamlit background engine...
⏳ Fetching active live domain...

🚀 YOUR WORKING DASHBOARD URL: https://maximum-chris-reduction-reveals.trycloudflare.com

Click the link above to test your dynamic dashboard!
